# Model Comparison — Dividend Growth Signal

Quick comparison of regression heads on the dividend growth series.
All models share the same frozen FinBERT embeddings + Shiller macro features
from the main pipeline.  Minimal tuning — designed for fast exploration.

**Models compared:**
1. Ridge
2. PLS (Partial Least Squares)
3. PLS → Ridge (supervised compression then linear regression)
4. PLS → Kernel Ridge (supervised compression then nonlinear regression)
5. Random Forest
6. Gaussian Process (RBF + linear composite kernel)

**Evaluation:** walk-forward expanding window, same scheme as main pipeline.
All models use the same `Z` and `y` from `all_results["dividend_growth"]`
— no re-encoding required.  Run the main pipeline notebook first.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.linear_model import Ridge, RidgeCV
from sklearn.cross_decomposition import PLSRegression
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, DotProduct, WhiteKernel, ConstantKernel
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import mean_squared_error

SEED = 42
np.random.seed(SEED)

## Load data from main pipeline

`all_results` must be defined (run the main pipeline notebook first, or paste
the path to a saved `walkforward_predictions.csv` and `model_final.pkl`).

In [ ]:
# Pull dividend growth data from main pipeline results
SERIES = "dividend_growth"

res       = all_results[SERIES]
Z_full    = res["Z"]          # (n_waves, 776) — news embedding + Shiller
y         = res["y"]          # (n_waves,) — median survey forecast
waves_df  = res["waves_df"]
min_train = res["min_train"]  # 30 for dividend_growth
n         = len(y)

N_MACRO       = len(SHILLER_FEATURES)   # 8
Z_news        = Z_full[:, :-N_MACRO]    # (n, 768) news only
Z_macro       = Z_full[:, -N_MACRO:]    # (n, 8)   Shiller only
feature_names = SHILLER_FEATURES

print(f"Series:       {SERIES}")
print(f"Total waves:  {n}")
print(f"Min train:    {min_train}")
print(f"Eval steps:   {n - min_train}")
print(f"Z_full shape: {Z_full.shape}")
print(f"Z_news shape: {Z_news.shape}")
print(f"Z_macro shape:{Z_macro.shape}")

## Walk-forward harness

In [ ]:
def walk_forward(Z, y, min_train, model_fn, scale=True):
    """
    Generic walk-forward expanding window.
    model_fn(Z_train, y_train) → fitted model with .predict(Z_val) method.
    scale=True: StandardScaler fitted on training set at each step.
    Returns dict with rmse, ar1_rmse, r2, dir_acc, wf_df.
    """
    records = []
    for t in range(min_train, len(y)):
        Z_tr, y_tr = Z[:t], y[:t]
        Z_va, y_va = Z[t:t+1], y[t:t+1]

        if scale:
            sc   = StandardScaler().fit(Z_tr)
            Z_tr = sc.transform(Z_tr)
            Z_va = sc.transform(Z_va)

        m     = model_fn(Z_tr, y_tr)
        y_hat = float(m.predict(Z_va).ravel()[0])
        records.append({"realized": float(y_va[0]), "predicted": y_hat,
                         "train_size": t})

    wf = pd.DataFrame(records)
    wf["error"]         = wf["realized"] - wf["predicted"]
    wf["sq_error"]      = wf["error"] ** 2
    wf["ar1_pred"]      = np.concatenate([[y[min_train-1]], wf["realized"].values[:-1]])
    wf["ar1_sq_error"]  = (wf["realized"] - wf["ar1_pred"]) ** 2
    wf["wave_date"]     = waves_df["wave_date"].values[min_train:]

    rmse     = np.sqrt(wf["sq_error"].mean())
    ar1_rmse = np.sqrt(wf["ar1_sq_error"].mean())
    y_wf     = wf["realized"].values
    r2       = 1 - wf["sq_error"].sum() / np.sum((y_wf - y_wf.mean())**2)
    dy_t     = np.diff(y_wf)
    dy_p     = np.diff(wf["predicted"].values)
    dir_acc  = np.mean(np.sign(dy_t) == np.sign(dy_p)) if len(dy_t) > 0 else np.nan
    return {"rmse": rmse, "ar1_rmse": ar1_rmse, "r2": r2,
            "dir_acc": dir_acc, "wf_df": wf}


def tune_alpha_gcv(Z_train, y_train, alphas=None):
    """Select Ridge α via GCV on training set (fast, no per-fold R² issue)."""
    if alphas is None:
        alphas = np.logspace(-3, 6, 40)
    sc  = StandardScaler().fit(Z_train)
    Zs  = sc.transform(Z_train)
    rc  = RidgeCV(alphas=alphas, gcv_mode="auto",
                  scoring="neg_mean_squared_error")
    rc.fit(Zs, y_train)
    return rc.alpha_


def tune_pls_components(Z_train, y_train, max_k=10):
    """Select PLS n_components by LOO-CV MSE on training set."""
    loo    = LeaveOneOut()
    max_k  = min(max_k, Z_train.shape[0] - 2, Z_train.shape[1])
    best_k, best_mse = 1, np.inf
    for k in range(1, max_k + 1):
        preds = np.empty(len(y_train))
        for tr, va in loo.split(Z_train):
            m = PLSRegression(n_components=k, scale=True)
            m.fit(Z_train[tr], y_train[tr])
            preds[va] = m.predict(Z_train[va]).ravel()
        mse = mean_squared_error(y_train, preds)
        if mse < best_mse:
            best_mse, best_k = mse, k
    return best_k


# Tune shared hyperparameters once on initial training window
print("Tuning on initial training window...")
Z_init, y_init = Z_full[:min_train], y[:min_train]
Z_news_init    = Z_news[:min_train]

alpha_ridge       = tune_alpha_gcv(Z_init, y_init)
alpha_ridge_news  = tune_alpha_gcv(Z_news_init, y_init)
best_pls_k        = tune_pls_components(Z_news_init, y_init, max_k=8)

print(f"  Ridge α (news+macro):  {alpha_ridge:.2e}")
print(f"  Ridge α (news only):   {alpha_ridge_news:.2e}")
print(f"  PLS n_components:      {best_pls_k}")

## Model definitions

In [ ]:
# ── 1. Ridge (news + macro) ───────────────────────────────────────────────────
def model_ridge(Z_tr, y_tr):
    return Ridge(alpha=alpha_ridge).fit(Z_tr, y_tr)

# ── 2. PLS (news embeddings only) ────────────────────────────────────────────
def model_pls(Z_tr, y_tr):
    # PLS has its own scaling so scale=False in walk_forward
    return PLSRegression(n_components=best_pls_k, scale=True).fit(Z_tr, y_tr)

# ── 3. PLS → Ridge stack ──────────────────────────────────────────────────────
# PLS projects embeddings to k components; Ridge then regresses on those + macro.
class PLSRidgeStack:
    def __init__(self, n_components, alpha):
        self.pls   = PLSRegression(n_components=n_components, scale=True)
        self.ridge = Ridge(alpha=alpha)
        self.n_mac = N_MACRO

    def fit(self, Z_tr, y_tr):
        # Split news and macro
        Z_news_tr  = Z_tr[:, :-self.n_mac] if self.n_mac > 0 else Z_tr
        Z_macro_tr = Z_tr[:, -self.n_mac:] if self.n_mac > 0 else np.empty((len(Z_tr), 0))
        # PLS on news
        self.pls.fit(Z_news_tr, y_tr)
        Z_pls = self.pls.transform(Z_news_tr)
        # Ridge on PLS components + macro
        Z_combined = np.hstack([Z_pls, Z_macro_tr]) if self.n_mac > 0 else Z_pls
        self.sc_combined = StandardScaler().fit(Z_combined)
        self.ridge.fit(self.sc_combined.transform(Z_combined), y_tr)
        return self

    def predict(self, Z_va):
        Z_news_va  = Z_va[:, :-self.n_mac] if self.n_mac > 0 else Z_va
        Z_macro_va = Z_va[:, -self.n_mac:] if self.n_mac > 0 else np.empty((len(Z_va), 0))
        Z_pls      = self.pls.transform(Z_news_va)
        Z_combined = np.hstack([Z_pls, Z_macro_va]) if self.n_mac > 0 else Z_pls
        return self.ridge.predict(self.sc_combined.transform(Z_combined))

def model_pls_ridge(Z_tr, y_tr):
    return PLSRidgeStack(best_pls_k, alpha_ridge).fit(Z_tr, y_tr)

# ── 4. PLS → Kernel Ridge stack ───────────────────────────────────────────────
# Same PLS compression, then RBF Kernel Ridge for nonlinearity in latent space.
class PLSKernelRidgeStack:
    def __init__(self, n_components, alpha=1.0, gamma=None):
        self.pls   = PLSRegression(n_components=n_components, scale=True)
        self.kr    = KernelRidge(kernel="rbf", alpha=alpha, gamma=gamma)
        self.n_mac = N_MACRO

    def fit(self, Z_tr, y_tr):
        Z_news_tr  = Z_tr[:, :-self.n_mac] if self.n_mac > 0 else Z_tr
        Z_macro_tr = Z_tr[:, -self.n_mac:] if self.n_mac > 0 else np.empty((len(Z_tr), 0))
        self.pls.fit(Z_news_tr, y_tr)
        Z_pls = self.pls.transform(Z_news_tr)
        Z_combined = np.hstack([Z_pls, Z_macro_tr]) if self.n_mac > 0 else Z_pls
        self.sc = StandardScaler().fit(Z_combined)
        self.kr.fit(self.sc.transform(Z_combined), y_tr)
        return self

    def predict(self, Z_va):
        Z_news_va  = Z_va[:, :-self.n_mac] if self.n_mac > 0 else Z_va
        Z_macro_va = Z_va[:, -self.n_mac:] if self.n_mac > 0 else np.empty((len(Z_va), 0))
        Z_pls      = self.pls.transform(Z_news_va)
        Z_combined = np.hstack([Z_pls, Z_macro_va]) if self.n_mac > 0 else Z_pls
        return self.kr.predict(self.sc.transform(Z_combined))

def model_pls_kr(Z_tr, y_tr):
    # gamma=None → sklearn default: 1/n_features in latent space
    return PLSKernelRidgeStack(best_pls_k, alpha=0.1, gamma=None).fit(Z_tr, y_tr)

# ── 5. Random Forest ──────────────────────────────────────────────────────────
def model_rf(Z_tr, y_tr):
    # Use tuned params from main pipeline if available, else sensible defaults
    try:
        bp  = all_results[SERIES]["best_params"]
        msl = bp["min_samples_leaf"]
        mf  = bp["max_features"]
    except Exception:
        msl, mf = 3, 0.10
    return RandomForestRegressor(
        n_estimators=300, min_samples_leaf=msl,
        max_features=mf, random_state=SEED, n_jobs=-1
    ).fit(Z_tr, y_tr)

# ── 6. Gaussian Process ───────────────────────────────────────────────────────
# RBF (nonlinear) + DotProduct (linear) + WhiteKernel (noise).
# Applied on PLS components to keep dimensionality tractable.
class PLSGaussianProcess:
    def __init__(self, n_components):
        self.pls   = PLSRegression(n_components=n_components, scale=True)
        kernel     = (ConstantKernel(1.0) * RBF(length_scale=1.0)
                      + DotProduct(sigma_0=1.0)
                      + WhiteKernel(noise_level=0.1))
        self.gp    = GaussianProcessRegressor(
            kernel=kernel, n_restarts_optimizer=3,
            normalize_y=True, random_state=SEED
        )
        self.n_mac = N_MACRO

    def fit(self, Z_tr, y_tr):
        Z_news_tr  = Z_tr[:, :-self.n_mac] if self.n_mac > 0 else Z_tr
        Z_macro_tr = Z_tr[:, -self.n_mac:] if self.n_mac > 0 else np.empty((len(Z_tr), 0))
        self.pls.fit(Z_news_tr, y_tr)
        Z_pls = self.pls.transform(Z_news_tr)
        Z_combined = np.hstack([Z_pls, Z_macro_tr]) if self.n_mac > 0 else Z_pls
        self.sc = StandardScaler().fit(Z_combined)
        self.gp.fit(self.sc.transform(Z_combined), y_tr)
        return self

    def predict(self, Z_va):
        Z_news_va  = Z_va[:, :-self.n_mac] if self.n_mac > 0 else Z_va
        Z_macro_va = Z_va[:, -self.n_mac:] if self.n_mac > 0 else np.empty((len(Z_va), 0))
        Z_pls      = self.pls.transform(Z_news_va)
        Z_combined = np.hstack([Z_pls, Z_macro_va]) if self.n_mac > 0 else Z_pls
        return self.gp.predict(self.sc.transform(Z_combined))

def model_gp(Z_tr, y_tr):
    return PLSGaussianProcess(best_pls_k).fit(Z_tr, y_tr)

print("All model definitions ready.")

## Run all models

In [ ]:
MODELS = {
    "Ridge":           (model_ridge,     Z_full, True),
    "PLS":             (model_pls,       Z_news, False),  # PLS has internal scaling
    "PLS → Ridge":     (model_pls_ridge, Z_full, False),  # handles scaling internally
    "PLS → KernelRidge": (model_pls_kr, Z_full, False),
    "Random Forest":   (model_rf,        Z_full, False),  # RF doesn't need scaling
    "PLS → GP":        (model_gp,        Z_full, False),
}

results = {}
for name, (fn, Z_in, do_scale) in MODELS.items():
    print(f"Running {name}...", end=" ", flush=True)
    res = walk_forward(Z_in, y, min_train, fn, scale=do_scale)
    results[name] = res
    print(f"RMSE={res['rmse']:.4f}  R²={res['r2']:.3f}  DirAcc={res['dir_acc']:.1%}")

In [ ]:
ar1_rmse = list(results.values())[0]["ar1_rmse"]

print(f"\n{'─'*65}")
print(f"{'Model':<22} {'RMSE':>8} {'vs AR(1)':>10} {'R²':>8} {'DirAcc':>9}")
print(f"{'─'*65}")
print(f"{'AR(1) baseline':<22} {ar1_rmse:>8.4f} {'—':>10} {'0.000':>8} {'—':>9}")
for name, res in results.items():
    delta = res['rmse'] - ar1_rmse
    sign  = "+" if delta >= 0 else ""
    print(f"{name:<22} {res['rmse']:>8.4f} {sign+f'{delta:.4f}':>10} "
          f"{res['r2']:>8.3f} {res['dir_acc']:>8.1%}")
print(f"{'─'*65}")

In [ ]:
model_names = list(results.keys())
colors = ["#2c5f8a", "#e07b39", "#4a9e6b", "#9b59b6", "#c0392b", "#16a085"]
color_map = dict(zip(model_names, colors))

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle(f"Model Comparison — {SERIES.replace('_',' ').title()}  "
             f"(n_eval={n - min_train}, PLS k={best_pls_k})",
             fontweight="bold", fontsize=12)

# Panel 1: RMSE bar chart
ax = axes[0]
names_bar = ["AR(1)"] + model_names
rmses     = [ar1_rmse] + [results[m]["rmse"] for m in model_names]
bar_colors= ["#7f8c8d"] + [color_map[m] for m in model_names]
bars = ax.bar(names_bar, rmses, color=bar_colors, alpha=0.88, edgecolor="white", width=0.6)
ax.axhline(ar1_rmse, color="#7f8c8d", ls="--", lw=1, alpha=0.6)
for bar, val in zip(bars, rmses):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.0003,
            f"{val:.4f}", ha="center", va="bottom", fontsize=7.5)
ax.set_ylabel("Walk-forward RMSE"); ax.set_title("RMSE (lower = better)")
ax.set_xticklabels(names_bar, rotation=30, ha="right", fontsize=8)
ax.grid(alpha=0.25, axis="y")

# Panel 2: R² bar chart
ax = axes[1]
r2s = [0.0] + [results[m]["r2"] for m in model_names]
bars = ax.bar(names_bar, r2s, color=bar_colors, alpha=0.88, edgecolor="white", width=0.6)
ax.axhline(0, color="black", lw=0.8, ls="--")
for bar, val in zip(bars, r2s):
    offset = 0.01 if val >= 0 else -0.04
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + offset,
            f"{val:.3f}", ha="center", va="bottom", fontsize=7.5)
ax.set_ylabel("Walk-forward R²"); ax.set_title("R²  (AR(1) = 0 by construction)")
ax.set_xticklabels(names_bar, rotation=30, ha="right", fontsize=8)
ax.grid(alpha=0.25, axis="y")

# Panel 3: walk-forward time series
ax = axes[2]
dates = waves_df["wave_date"].values[min_train:]
ax.plot(dates, y[min_train:], "-", color="black", lw=2, label="Realized", zorder=6)
ax.axhline(y[min_train-1], color="#7f8c8d", ls=":", lw=1, alpha=0.7, label="AR(1)")
for name, res in results.items():
    ax.plot(dates, res["wf_df"]["predicted"].values, "--",
            color=color_map[name], alpha=0.75, lw=1.2, label=name)
ax.set_xlabel("Wave date"); ax.set_ylabel("Forecast")
ax.set_title("Walk-forward predictions")
ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig("model_comparison_dividend.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved model_comparison_dividend.png")

## GP uncertainty bands

The Gaussian Process model gives calibrated uncertainty estimates per prediction.
This cell plots the walk-forward predictions with ±1σ bands — useful for
identifying which quarters the GP is least confident about.

In [ ]:
# Rerun GP walk-forward collecting std predictions
gp_records = []
for t in range(min_train, n):
    Z_tr, y_tr = Z_full[:t], y[:t]
    Z_va       = Z_full[t:t+1]
    gp_model   = PLSGaussianProcess(best_pls_k).fit(Z_tr, y_tr)

    # Need to replicate internal transform for std prediction
    Z_news_va  = Z_va[:, :-N_MACRO]
    Z_macro_va = Z_va[:, -N_MACRO:]
    Z_pls      = gp_model.pls.transform(Z_news_va)
    Z_combined = np.hstack([Z_pls, Z_macro_va])
    Z_sc       = gp_model.sc.transform(Z_combined)

    y_mean, y_std = gp_model.gp.predict(Z_sc, return_std=True)
    gp_records.append({
        "wave_date": waves_df.iloc[t]["wave_date"],
        "realized":  float(y[t]),
        "predicted": float(y_mean[0]),
        "std":       float(y_std[0]),
    })

gp_df = pd.DataFrame(gp_records)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(gp_df["wave_date"], gp_df["realized"],  "-",  color="black", lw=2,
        label="Realized")
ax.plot(gp_df["wave_date"], gp_df["predicted"], "--", color="#16a085", lw=1.5,
        label="GP predicted")
ax.fill_between(gp_df["wave_date"],
                gp_df["predicted"] - gp_df["std"],
                gp_df["predicted"] + gp_df["std"],
                alpha=0.2, color="#16a085", label="±1σ uncertainty")
ax.set_xlabel("Wave date"); ax.set_ylabel("Dividend growth expectation")
ax.set_title("Gaussian Process walk-forward predictions with uncertainty bands")
ax.legend(fontsize=9); ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("gp_uncertainty_dividend.png", dpi=150, bbox_inches="tight")
plt.show()

# Flag high-uncertainty quarters
high_unc = gp_df[gp_df["std"] > gp_df["std"].quantile(0.75)].copy()
print(f"\nHigh-uncertainty quarters (top 25% σ):")
for _, row in high_unc.iterrows():
    print(f"  {pd.Timestamp(row['wave_date']).strftime('%Y-%m')}  "
          f"σ={row['std']:.4f}  realized={row['realized']:.4f}  "
          f"predicted={row['predicted']:.4f}")

## Directional accuracy detail

For the dividend series, directional accuracy may be more meaningful than RMSE
given the short evaluation window.  This cell shows per-model directional
accuracy alongside a breakdown of which quarters each model got right.

In [ ]:
print(f"{'─'*55}")
print(f"{'Model':<22} {'DirAcc':>9} {'Correct':>9} {'Wrong':>9}")
print(f"{'─'*55}")

dy_true = np.diff(y[min_train:])
for name, res in results.items():
    dy_pred = np.diff(res["wf_df"]["predicted"].values)
    correct = np.sum(np.sign(dy_true) == np.sign(dy_pred))
    wrong   = len(dy_true) - correct
    pct     = correct / len(dy_true)
    print(f"{name:<22} {pct:>8.1%} {correct:>9} {wrong:>9}")
print(f"{'─'*55}")
print(f"\nTotal evaluation steps: {n - min_train}")
print(f"Direction pairs:        {len(dy_true)}")